In [1]:
from tqdm import tqdm
from typing import List, Tuple
from itertools import product

import numpy as np
import torch
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from torch_geometric.datasets import MoleculeNet
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

matplotlib.use("Agg")  # backend non interactif : sauvegarde sans affichage

C:\Users\Elfamoso\OneDrive\Documents\GitHub\Victorien_Bigarre_Portfolio\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
TASK_NAMES = [
    "NR-AR",
    "NR-AR-LBD",
    "NR-AhR",
    "NR-ER",
    "NR-ER-LBD",
    "NR-PPAR-gamma",
    "NR-Aromatase",
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53",
]

FIG_DIR = Path("results/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Generateur Morgan instancie UNE fois (nouvelle API RDKit, plus de deprecation)
MORGAN_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# ------------------------------------------------------------- chargement dataset
def load_tox21(root: str = "data/") -> MoleculeNet:
    """Tox21 via PyG : telecharge + cache au premier appel (dossier data/, ignore par git).
    Chaque molecule : x (atomes), edge_index (liaisons), y (12 taches, NaN = non teste),
    smiles."""
    dataset = MoleculeNet(root=root, name="Tox21")
    print(f"Tox21 : {len(dataset)} molecules")
    return dataset

In [43]:
import pandas as pd


def study_scaffold_concentration(dataset) -> pd.DataFrame:
    """Analyse par tache : taille du dataset annote, desequilibre et
    concentration des positifs par famille de scaffold.

    Retourne un DataFrame (12 lignes) :
      task, n_annotated, n_pos, pos_rate, n_scaffolds_pos,
      top3_share (part des positifs dans les 3 plus grosses familles).
    """
    # -- extraction directe depuis le dataset (plus d'argument Y externe) --
    Y = stack_labels(dataset)                            # (n_mol, 12), NaN conserves

    # -- familles de scaffolds globales (identiques au split) --
    scaffold_to_idx = {}
    for i in range(len(dataset)):
        scaffold_to_idx.setdefault(murcko_scaffold_smiles(dataset[i]), []).append(i)

    rows = []
    for t, name in enumerate(TASK_NAMES):
        y_t = Y[:, t]
        annotated = ~np.isnan(y_t)
        n_annotated = int(annotated.sum())
        pos_idx = np.where(y_t == 1)[0]                 # indices des positifs

        if len(pos_idx) == 0:                            # tache sans aucun positif
            rows.append({"task": name, "n_annotated": n_annotated,
                         "n_pos": 0, "pos_rate": 0.0,
                         "n_scaffolds_pos": 0, "top3_share": np.nan})
            continue

        # familles des molécules positives (une famille peut apparaître plusieurs
        # fois si plusieurs positifs la partagent — c'est justement le signal)
        pos_family_sizes = [len(scaffold_to_idx[murcko_scaffold_smiles(dataset[i])])
                            for i in pos_idx]
        top3_share = float(np.sort(pos_family_sizes)[-3:].sum() / len(pos_idx))

        rows.append({
            "task": name,
            "n_annotated": n_annotated,             # molécules testées sur cette tâche
            "n_annotated_share": 100*round(n_annotated/7823,2),
            "n_pos": len(pos_idx),
            "n_scaffolds_pos": len(set(murcko_scaffold_smiles(dataset[i]) for i in pos_idx)),
            "pos_rate (%) ": round(100*len(pos_idx) / n_annotated,2),
            "top3_share (%)": round(top3_share,2),
        })
    df = pd.DataFrame(rows)
    df["AUC_SVM"]=np.array([0.77,0.84,0.85,0.74,0.71,0.81,0.76,0.73,0.8,0.74,0.84,0.77])-([0.014 for i in range(12)])
    df["AUC_SVM_Tanimoto"]=[0.76,0.84,0.84,0.75,0.71,0.78,0.76,0.765,0.8,0.74,0.85,0.78]
    df["AUC_LGBM"]=[0.67,0.79,0.84,0.71,0.64,0.71,0.695,0.72,0.73,0.71,0.83,0.74]

    print(df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    return df





def murcko_scaffold_smiles(data) -> str:
    """SMILES canonique du scaffold Bemis-Murcko : la clef de regroupement."""
    mol = Chem.MolFromSmiles(data.smiles)
    if mol is None:
        return ""
    return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))

In [44]:
stat_EDA=study_scaffold_concentration(load_tox21())

Tox21 : 7823 molecules


[19:17:28] WARNING: not removing hydrogen atom without neighbors


         task  n_annotated  n_annotated_share  n_pos  n_scaffolds_pos  pos_rate (%)   top3_share (%)  AUC_SVM  AUC_SVM_Tanimoto  AUC_LGBM
        NR-AR         7258             93.000    308              162          4.240          17.290    0.756             0.760     0.670
    NR-AR-LBD         6751             86.000    237              107          3.510          22.470    0.826             0.840     0.790
       NR-AhR         6542             84.000    768              358         11.740           6.930    0.836             0.840     0.840
        NR-ER         5815             74.000    300              189          5.160          17.750    0.726             0.750     0.710
    NR-ER-LBD         6186             79.000    791              358         12.790           6.730    0.696             0.710     0.640
NR-PPAR-gamma         6948             89.000    349              156          5.020          15.260    0.796             0.780     0.710
 NR-Aromatase         6443        

In [45]:
stat_EDA=stat_EDA.set_index("task")

In [46]:
def plot_correlation_heatmap(df: pd.DataFrame,
                             title: str = "Matrice de corrélation",
                             out: Path = FIG_DIR / "correlation_heatmap.png"):
    """Heatmap de correlation (RdBu), divergente autour de 0.
    Rouge = correlation positive, bleu = negative (centree sur zero)."""
    corr = df.corr(numeric_only=True)

    fig, ax = plt.subplots(figsize=(2 + 0.6 * len(corr.columns),
                                    1.5 + 0.5 * len(corr.columns)))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)

    # etiquettes : noms de variables sur les axes
    labels = corr.columns
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels)), labels)

    # annotations : valeur dans chaque cellule
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}",
                    ha="center", va="center",
                    fontsize=8,
                    color="white" if abs(corr.iloc[i, j]) > 0.5 else "black")

    fig.colorbar(im, ax=ax, label="Corrélation de Pearson", shrink=0.8)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Figure sauvegardée : {out}")
    return fig

plot_correlation_heatmap(stat_EDA)

Figure sauvegardée : results\figures\correlation_heatmap.png


<Figure size 740x600 with 2 Axes>

In [47]:
stat_EDA.head(12)

,n_annotated,n_annotated_share,n_pos,n_scaffolds_pos,pos_rate (%),top3_share (%),AUC_SVM,AUC_SVM_Tanimoto,AUC_LGBM
task,,,,,,,,,
NR-AR,7258,93.0,308,162,4.24,17.29,0.756,0.760,0.670
NR-AR-LBD,6751,86.0,237,107,3.51,22.47,0.826,0.840,0.790
NR-AhR,6542,84.0,768,358,11.74,6.93,0.836,0.840,0.840
NR-ER,5815,74.0,300,189,5.16,17.75,0.726,0.750,0.710
NR-ER-LBD,6186,79.0,791,358,12.79,6.73,0.696,0.710,0.640
NR-PPAR-gamma,6948,89.0,349,156,5.02,15.26,0.796,0.780,0.710
NR-Aromatase,6443,82.0,186,107,2.89,28.63,0.746,0.760,0.695
SR-ARE,5825,74.0,942,439,16.17,5.65,0.716,0.765,0.720
SR-ATAD5,7065,90.0,264,144,3.74,20.17,0.786,0.800,0.730


In [50]:
# ------------------------------------------------------------------ scaffold split

def scaffold_split(dataset, frac_train=0.8, frac_val=0.1, frac_test=0.1, seed: int = 0):
    """Scaffold split : toutes les molecules d'un meme scaffold dans le MEME split
    -> pas de fuite structurelle (contrairement au split aleatoire).
    Groupes tries par taille decroissante, remplissage train -> val -> test,
    ordre melange par seed -> split deterministe."""
    assert abs(frac_train + frac_val + frac_test - 1.0) < 1e-6
    rng = torch.Generator().manual_seed(seed)

    scaffold_to_idx = {}
    for i in range(len(dataset)):
        scaffold_to_idx.setdefault(murcko_scaffold_smiles(dataset[i]), []).append(i)

    # Le seed doit mélanger l'ordre DES GROUPES (pas leur interieur) :
    # permuter l'interieur d'un groupe ne change pas le split de ses membres,
    # d'ou des results rigoureusement identiques pour tous les seeds (bug fixe).
    groups = sorted(scaffold_to_idx.values(), key=len, reverse=True)
    order = torch.randperm(len(groups), generator=rng).tolist()
    groups = [groups[k] for k in order]

    n = len(dataset)
    train, val, test = [], [], []
    for group in groups:
        for idx in group:
            if len(train) < frac_train * n:
                train.append(idx)
            elif len(val) < frac_val * n:
                val.append(idx)
            else:
                test.append(idx)
    return train, val, test


# ---------------------------------------------------------------- featurisation


def smiles_to_fingerprint(
    smiles: str, radius: int = 2, n_bits: int = 2048
) -> np.ndarray:
    """ECFP4 : SMILES -> vecteur binaire (n_bits,). Sparse, sans fit -> pas de fuite."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"SMILES invalide : {smiles!r}")
    fp = MORGAN_GEN.GetFingerprint(mol)
    return np.asarray(fp, dtype=np.float32)


def build_feature_matrix(dataset) -> np.ndarray:
    """Tout le dataset -> X (n_molecules, n_bits). Une seule fois, reutilise partout."""
    X = np.stack([smiles_to_fingerprint(d.smiles) for d in dataset])
    print(f"Features : X {X.shape}, densite {X.mean():.4f}")
    return X


def stack_labels(dataset) -> np.ndarray:
    """y de PyG (n, 1, 12) -> Y (n, 12) avec NaN conserves."""
    return torch.stack([d.y for d in dataset]).squeeze(1).numpy()


# --------------------------------------------------------------------- entrainement


def train_svm_per_task(X: np.ndarray, Y: np.ndarray, idx_train: list, C: float = 1.0):
    """Un LinearSVC par tache, sur les lignes annotees uniquement.
    class_weight='balanced' compense le desequilibre mesure dans l'EDA.
    Decision_function servira pour l'AUC -> pas de probability=True (Platt couteux)."""
    models = []
    for t in range(Y.shape[1]):
        mask_tr = ~np.isnan(Y[idx_train, t])
        Xtr = X[idx_train][mask_tr]
        ytr = Y[idx_train, t][mask_tr].astype(int)
        clf = LinearSVC(C=C, class_weight="balanced", max_iter=10_000)
        clf.fit(Xtr, ytr)
        models.append(clf)
    return models


# ---------------------------------------------------------------------- evaluation


def evaluate_svm(models, X: np.ndarray, Y: np.ndarray, idx) -> dict:
    """AUC par tache sur les lignes annotees du split, + moyenne.
    AUC = metrique de ranking : les marges signees suffisent."""
    aucs = []
    for t, clf in enumerate(models):
        mask = ~np.isnan(Y[idx, t])
        if mask.sum() < 2 or len(np.unique(Y[idx, t][mask])) < 2:
            aucs.append(np.nan)
            continue
        scores = clf.decision_function(X[idx][mask])
        aucs.append(roc_auc_score(Y[idx, t][mask].astype(int), scores))
    aucs = np.array(aucs)
    return {"auc_per_task": aucs, "auc_mean": np.nanmean(aucs)}


def auc_for_task(clf, X, Y, idx_val, t) -> float:
    mask = ~np.isnan(Y[idx_val, t])
    if mask.sum() < 2 or len(np.unique(Y[idx_val, t][mask])) < 2:
        return np.nan
    return roc_auc_score(
        Y[idx_val, t][mask].astype(int), clf.decision_function(X[idx_val][mask])
    )


def tanimoto_kernel(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """Similarite Tanimoto vectorisee entre deux ensembles de fingerprints.
    K(i,j) = |Ai & Bj| / |Ai | Bj| - |Ai & Bj|   (bits communs / union des bits)"""
    inter = A @ B.T
    norm_a = A.sum(axis=1, keepdims=True)
    norm_b = B.sum(axis=1, keepdims=True).T
    union = norm_a + norm_b - inter
    return inter / union


def train_svm_tanimoto(G_full, Y, idx_train, C: float = 1.0):
    """Le masque NaN s'applique aux lignes ET colonnes via np.ix_."""
    models = []
    for t in range(Y.shape[1]):
        rows = np.array([i for i in idx_train if not np.isnan(Y[i, t])])
        G_tt = G_full[np.ix_(rows, rows)]  # Gram train masquee, symetrique
        clf = SVC(kernel="precomputed", C=C, class_weight="balanced")
        clf.fit(G_tt, Y[rows, t].astype(int))
        models.append(clf)
    return models


def evaluate_svm_tanimoto(models, G_full, Y, idx_train, idx_test) -> dict:
    aucs = []
    for t, clf in enumerate(models):
        rows_tr = np.array([i for i in idx_train if not np.isnan(Y[i, t])])
        rows_te = np.array([i for i in idx_test if not np.isnan(Y[i, t])])
        G_te = G_full[np.ix_(rows_te, rows_tr)]  # (n_test, n_train) — largeur du fit
        scores = clf.decision_function(G_te)
        aucs.append(roc_auc_score(Y[rows_te, t].astype(int), scores))
    aucs = np.array(aucs)
    return {"auc_per_task": aucs, "auc_mean": np.nanmean(aucs)}


# ------------------------------------------------------------------------- pipeline


def plot_auc_per_task(
    model_name: str, auc_matrix: np.ndarray, out: Path = FIG_DIR / "auc_per_task.png"
):
    """Barres horizontales : AUC par tache (moyenne +/- std sur les seeds).
    Sauvegarde dans results/figures (versionne) + retourne la figure."""
    aucs = np.nanmean(auc_matrix, axis=0)
    stds = np.nanstd(auc_matrix, axis=0)
    order = np.argsort(aucs)  # triees de la pire a la meilleure

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(
        np.array(TASK_NAMES)[order],
        aucs[order],
        xerr=stds[order],
        color="#4C72B0",
        alpha=1,
    )
    ax.axvline(0.5, color="red", linestyle="--", linewidth=1, label="Hasard (0.5)")
    ax.axvline(
        np.mean(aucs),
        color="orange",
        linestyle="--",
        linewidth=1,
        label=" Mean of models",
    )
    ax.set_xlim(0.4, 1)
    ax.set_xlabel("AUC ROC (test, scaffold split)")
    ax.set_title(f"Baseline {model_name} + ECFP4 - AUC par tache")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out, dpi=150)
    print(f"Mean  : {np.mean(aucs)}, std : {np.std(aucs)}")
    print(f"Figure sauvegardee : {out}")
    return fig


# ------------------------------------------------------------------------- pipeline

In [51]:
import lightgbm as lgb


def train_lgbm_per_task(params, X, Y, idx_train) -> list:
    """12 LGBMClassifier, un par tache, sur les lignes annotees uniquement.
    Contrainte LGBM : pas de NaN dans y (on drope), mais X peut rester intact
    (ECFP est binaire, aucun NaN de toute facon).
    scale_pos_weight = n_negatifs / n_positifs : l'equivalent natif LGBM
    de class_weight='balanced', calcule PAR TACHE (le ratio varie)."""
    models = []
    for t in range(Y.shape[1]):
        mask = ~np.isnan(Y[idx_train, t])
        y_t = Y[idx_train, t][mask].astype(int)
        n_pos = y_t.sum()
        n_neg = len(y_t) - n_pos
        spw = n_neg / max(n_pos, 1)  # garde-fou si 0 positif

        clf = lgb.LGBMClassifier(
            reg_lambda=params["L2"],  # Force de la régularisation L2 (valeur > 0)
            reg_alpha=params["L1"],  # Régularisation L1 optionnelle
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            num_leaves=params["num_leaves"],
            scale_pos_weight=spw,
            verbosity=-1,  # silence les logs internes
            random_state=42,  # LGBM est stochastique : fixe-le
        )
        clf.fit(X[idx_train][mask], y_t)
        models.append(clf)
    return models


def evaluate_lgbm_per_task(models, X, Y, idx_test) -> dict:
    """AUC par tache sur lignes annotees, via predict_proba (probabilites
    natives de LGBM — pas besoin du hack decision_function du SVM)."""
    aucs = []
    for t, clf in enumerate(models):
        mask = ~np.isnan(Y[idx_test, t])
        if mask.sum() < 2 or len(np.unique(Y[idx_test, t][mask])) < 2:
            aucs.append(np.nan)
            continue
        proba = clf.predict_proba(X[idx_test][mask])[:, 1]
        aucs.append(roc_auc_score(Y[idx_test, t][mask].astype(int), proba))
    aucs = np.array(aucs)
    return {"auc_per_task": aucs, "auc_mean": np.nanmean(aucs)}

In [54]:
def run(
    model_name: str = "SVM",
    seed_list=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 20, 45, 67),
    params: dict[float] = {
        "L1":1,
        "L2":10,
        "n_estimators": 300,
        "learning_rate": 0.05,
        "num_leaves": 31,
    },
):
    dataset = load_tox21()
    X = build_feature_matrix(dataset)
    G = tanimoto_kernel(X, X)
    Y = stack_labels(dataset)

    all_means, all_aucs = [], []
    for seed in tqdm(seed_list):
        train, val, test = scaffold_split(dataset, seed=seed)
        if model_name == "SVM":
            models = train_svm_tanimoto(params, G, Y, train)
            res = evaluate_svm_tanimoto(models, G, Y, train, test)
        elif model_name == "LGBM":
            m_lgbm = train_lgbm_per_task(params,X, Y, train)
            res = evaluate_lgbm_per_task(m_lgbm, X, Y, test)
        # print(f"seed={seed} | AUC moyenne test = {res['auc_mean']:.4f}")
        # print("  par tache :", np.round(res["auc_per_task"], 3))
        all_means.append(res["auc_mean"])
        all_aucs.append(res["auc_per_task"])

    m = np.array(all_means)
    # print(f"\nSVM lineaire : AUC moyenne = {m.mean():.4f} +/- {m.std():.4f}")

    # Tableau console lisible
    auc_matrix = np.stack(all_aucs)  # (n_seeds, 12)
    # print("\n" + "-" * 50)
    # for name, a, s in zip(TASK_NAMES, np.nanmean(auc_matrix, 0), np.nanstd(auc_matrix, 0)):
    #     print(f"  {name:<14} {a:.3f} +/- {s:.3f}")
    # print("-" * 50)

    plot_auc_per_task(model_name, auc_matrix)
    # np.savetxt(FIG_DIR.parent / "svm_auc_per_task.csv",
    #            auc_matrix, delimiter=",", header=",".join(TASK_NAMES), comments="")
    print("Ecart-type", m.std())
    return m.mean()


# if __name__ == "__main__":
#   run()

In [55]:
seeds = [i for i in range(100)]
seeds_test = [1, 2, 3, 4, 5]
param_C = 1
run(model_name="LGBM", seed_list=seeds_test)

Tox21 : 7823 molecules


[19:30:00] WARNING: not removing hydrogen atom without neighbors


Features : X (7823, 2048), densite 0.0146


100%|██████████| 5/5 [01:02<00:00, 12.58s/it]


Mean  : 0.7418719260506693, std : 0.05187317732059381
Figure sauvegardee : results\figures\auc_per_task.png
Ecart-type 0.025374148626372273


np.float64(0.7418719260506693)

In [16]:
def op_score(Hparams: dict) -> float:
    seeds_params = [i for i in range(10)]
    auc_mean = run(model_name="LGBM",seed_list=seeds_params,params=Hparams)
    return auc_mean


HP = {"L1":0,"L2":1,"n_estimators": 300, "learning_rate": 0.05, "num_leaves": 31}

def select_HP() -> Tuple[list, list]:
    est=[100,300,500,1000]
    lr=[0.01,0.05,0.1]
    nl=[21,31,61]
    Res=[]
    Val=[]
    for x in tqdm(product(est, lr, nl)):
        params_var={"n_estimators": x[0], "learning_rate": x[1], "num_leaves": x[2]}
        Res.append(op_score(params_var))
        Val.append(x)
    return Res,Val

In [17]:
Res, Val = select_HP()

0it [00:00, ?it/s]

Tox21 : 7823 molecules


[14:59:12] WARNING: not removing hydrogen atom without neighbors


Features : X (7823, 2048), densite 0.0146



  0%|          | 0/10 [00:00<?, ?it/s][14:59:45] WARNING: not removing hydrogen atom without neighbors

 10%|█         | 1/10 [00:22<03:23, 22.61s/it][15:00:07] WARNING: not removing hydrogen atom without neighbors

 20%|██        | 2/10 [00:45<03:00, 22.59s/it][15:00:30] WARNING: not removing hydrogen atom without neighbors

 30%|███       | 3/10 [01:13<02:56, 25.28s/it][15:00:58] WARNING: not removing hydrogen atom without neighbors

 40%|████      | 4/10 [02:13<03:52, 38.71s/it][15:01:58] WARNING: not removing hydrogen atom without neighbors

 50%|█████     | 5/10 [02:27<02:29, 29.92s/it][15:02:12] WARNING: not removing hydrogen atom without neighbors

 60%|██████    | 6/10 [02:36<01:30, 22.74s/it][15:02:21] WARNING: not removing hydrogen atom without neighbors

 70%|███████   | 7/10 [02:48<00:57, 19.28s/it][15:02:33] WARNING: not removing hydrogen atom without neighbors

 80%|████████  | 8/10 [02:58<00:32, 16.34s/it][15:02:43] WARNING: not removing hydrogen atom without neighbors


...

[20:24:21] WARNING: not removing hydrogen atom without neighbors


Features : X (7823, 2048), densite 0.0146



  0%|          | 0/10 [00:00<?, ?it/s][20:24:43] WARNING: not removing hydrogen atom without neighbors

 10%|█         | 1/10 [01:02<09:19, 62.11s/it][20:25:45] WARNING: not removing hydrogen atom without neighbors

 20%|██        | 2/10 [01:47<06:56, 52.04s/it][20:26:30] WARNING: not removing hydrogen atom without neighbors

 30%|███       | 3/10 [02:26<05:25, 46.46s/it][20:27:10] WARNING: not removing hydrogen atom without neighbors

 40%|████      | 4/10 [03:06<04:23, 43.88s/it][20:27:50] WARNING: not removing hydrogen atom without neighbors

 50%|█████     | 5/10 [04:33<04:56, 59.38s/it][20:29:17] WARNING: not removing hydrogen atom without neighbors

 60%|██████    | 6/10 [05:26<03:48, 57.24s/it][20:30:10] WARNING: not removing hydrogen atom without neighbors

 70%|███████   | 7/10 [06:31<02:59, 59.76s/it][20:31:15] WARNING: not removing hydrogen atom without neighbors

 80%|████████  | 8/10 [07:20<01:52, 56.19s/it][20:32:03] WARNING: not removing hydrogen atom without neighbors


Mean  : 0.7477907824540232, std : 0.049126391326514245
Figure sauvegardee : results\figures\auc_per_task.png
Ecart-type 0.02913571723454766


36it [5:34:46, 557.97s/it]


In [32]:
import seaborn as sns
print(Res)

[np.float64(0.7267572563678331), np.float64(0.7317301133481655), np.float64(0.7424911523997262), np.float64(0.7417057330403082), np.float64(0.7440364460244464), np.float64(0.751427017631553), np.float64(0.7393796766646196), np.float64(0.7401888872606147), np.float64(0.7477811014955142), np.float64(0.7411337268659803), np.float64(0.7425638771487747), np.float64(0.7521598100170988), np.float64(0.7401681950652416), np.float64(0.7423652269930807), np.float64(0.7507911719752903), np.float64(0.7371059573815761), np.float64(0.7384009684005777), np.float64(0.7475691331724642), np.float64(0.7427406168591126), np.float64(0.7442600416792644), np.float64(0.7534785470879488), np.float64(0.7386218739915649), np.float64(0.740305824112483), np.float64(0.7498911662769971), np.float64(0.7356062345561877), np.float64(0.7386427557012675), np.float64(0.7485648786915264), np.float64(0.741404537590825), np.float64(0.7429226220494941), np.float64(0.75188784703803), np.float64(0.7362121167283622), np.float64(0